# Stage 5 — extract

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and an inline eval. The implementation itself is yours to write.


## 1. Setup


Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running anything else in the same runtime), run them top-to-bottom.


### Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### Install dependencies


Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### Bridge your OpenAI key


Add `OPENAI_API_KEY` in Colab's Secrets panel (key icon, left sidebar) and toggle notebook access first.


In [ ]:
# OpenAI key bridge: Colab's userdata.get() does NOT populate os.environ,
# but our scripts read os.environ["OPENAI_API_KEY"]. Bridge it once.
# Add the key in Colab via the left sidebar → "Secrets" (key icon) → name it OPENAI_API_KEY.
import os
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
    if key:
        os.environ["OPENAI_API_KEY"] = key
        print("OPENAI_API_KEY set in os.environ")
    else:
        print("WARNING: OPENAI_API_KEY secret is empty — Stage 2/5 and *_llm.py evals will fail")
except Exception as e:
    print("Not running in Colab or userdata unavailable; set OPENAI_API_KEY yourself.")
    print("Detail:", e)


### Seed prior stages' outputs from the reference run


Stage 5 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_04/data/` from the canonical reference run so Stage 5 has inputs to work with — but only when the dir is empty, so re-running an earlier stage IN THIS runtime is not clobbered.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 5.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <5 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 5):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Configure inputs


The SCHEMA string is the extraction contract — what fields the LLM must produce, with type/enum hints. Edit MODEL or SCHEMA below; the cell also writes `schema.json` to the repo root for the standalone eval scripts.


In [ ]:
MODEL = "gpt-5.4-nano"

SCHEMA = '''\
{
  "pmid": "string",
  "source_type": "fulltext | abstract-only",
  "first_author": "string",
  "year": "integer",
  "mab_name": "string | null",
  "target": "string",
  "format": "IgG1 | IgG2 | IgG3 | IgG4 | bispecific | ADC | Fab | Fc-fusion | other",
  "development_stage": "discovery | lead optimization | IND-enabling | clinical translation | post-approval",
  "regulatory_context": "none-stated | IND-supporting | BLA-supporting | post-marketing",
  "threeRs_mentioned": "boolean",
  "author_reduction_recommendation": "string | null",
  "animal_arms": [
    {
      "species": "mouse | rat | cynomolgus | rhesus | dog | rabbit | minipig | other",
      "n_animals": "integer | null",
      "study_type": "PK | single-dose tox | repeat-dose tox | immunogenicity | biodistribution | efficacy | TCR",
      "duration_days": "integer | null",
      "species_justification": "pharmacological relevance | regulatory expectation | historical precedent | not stated",
      "cross_reactivity_evidence": "in-vitro binding shown | sequence homology only | not addressed",
      "endpoints_unique_to_animal": "string | null",
      "concurrent_nam": "string | null"
    }
  ],
  "nams_discussed": [
    {"method": "string", "context": "future work | limitation discussion | literature comparison"}
  ]
}
'''
with open("schema.json", "w") as f: f.write(SCHEMA)
print(f"MODEL={MODEL}  SCHEMA: {len(SCHEMA)} chars (also written to schema.json)")


## 3. Spec — paste this into Gemini


Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
For every `stage_04/data/*.md`, call OpenAI with MODEL and a prompt
that includes the SCHEMA string. Get back a JSON object per paper
matching the schema's shape.

The schema is a TYPE CONTRACT, not literal values — each field's
value is the TYPE of the data to extract, not the type label itself.

After the LLM returns each record:
  1. Force-overwrite `pmid` and `source_type` from the
     `stage_03/data/fetched.json` mapping (build a stem→pmid map and
     a stem→source_type map first). Never trust the LLM for IDs.
  2. Run a cheap hallucination check on `concurrent_nam`: any value
     whose substantive tokens (>4 chars) don't appear in the source
     text gets demoted to `nams_discussed` (keeps it visible but
     out of the structured arm).

Save each record to `stage_05/data/<stem>.json` (one file per paper).
```


## 4. Gotchas Gemini probably won't know


Copy any that apply into Gemini if it goes off-track:

- **No regex fallback for this stage.** It hard-fails without
  `OPENAI_API_KEY`. Bail with an assert.
- **JSON mode required.** `response_format={"type": "json_object"}`
  and `temperature=0`.
- **PMC stem → PMID** comes from `stage_03/data/fetched.json` — build
  the map up front and inject the pmid into the prompt so the LLM
  doesn't invent one.
- **Source type matters.** Tag each record `fulltext` (path ends
  `.pdf`/`.xml`) vs `abstract-only` (path ends `.json`). For
  abstract-only the LLM should return `[]` for `animal_arms` rather
  than guessing from a brief PubMed abstract.


## 5. Seed — a few lines to anchor Gemini


In [ ]:
import glob, json, os, time
assert os.environ.get("OPENAI_API_KEY"), \
    "extraction needs OPENAI_API_KEY (no fallback)"
from openai import OpenAI
client = OpenAI()
os.makedirs("stage_05/data", exist_ok=True)

with open("stage_03/data/fetched.json") as f:
    PMID_BY_PATH = json.load(f)
STEM_TO_PMID = {}
STEM_TO_SOURCE = {}
for pmid, path in PMID_BY_PATH.items():
    if not path: continue
    stem = os.path.splitext(os.path.basename(path))[0]
    STEM_TO_PMID[stem] = pmid
    STEM_TO_SOURCE[stem] = "fulltext" if path.endswith((".pdf", ".xml")) else "abstract-only"


## 6. Your implementation


Drive Gemini to fill this in. Iterate until the inspect cell below shows reasonable output and the eval cell passes.


In [ ]:
# TODO: implement Stage 5 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the inspect + eval cells next.


## 7. Inspect output


In [ ]:
import glob, json, os
SKIP = {"eval_script.json", "eval_llm.json", "score.json"}
extractions = []
for p in sorted(glob.glob("stage_05/data/*.json")):
    if os.path.basename(p) in SKIP: continue
    extractions.append((p, json.load(open(p))))
print(f"{len(extractions)} extractions in stage_05/data/\n")
total_arms = 0
for path, rec in extractions:
    arms = rec.get("animal_arms") or []
    total_arms += len(arms)
    stem = os.path.splitext(os.path.basename(path))[0]
    print(f"  {stem:18s}  pmid={rec.get('pmid','?'):10s}  "
          f"src={rec.get('source_type','?'):14s}  arms={len(arms)}")
    for a in arms[:2]:
        print(f"      • {a.get('species','?')} / {a.get('study_type','?')} "
              f"n={a.get('n_animals')} {a.get('duration_days','?')}d")
print(f"\nTotal animal_arms across papers: {total_arms}")


## 8. Run eval


Inline eval — same checks as `eval/eval_05_script.py`, but the code is right here so you can see what it's measuring. Writes `stage_05/eval/eval_script.json` + `score.json`.


In [ ]:
# Same checks as eval/eval_05_script.py, inlined.
import glob, json, os
os.makedirs("stage_05/eval", exist_ok=True)
ENUM_SOURCE = {"fulltext", "abstract-only"}
ENUM_FORMAT = {"IgG1","IgG2","IgG3","IgG4","bispecific","ADC","Fab","Fc-fusion","other"}
ENUM_STAGE  = {"discovery","lead optimization","IND-enabling","clinical translation","post-approval"}
ENUM_REG    = {"none-stated","IND-supporting","BLA-supporting","post-marketing"}
ENUM_SPECIES= {"mouse","rat","cynomolgus","rhesus","dog","rabbit","minipig","other"}
ENUM_STUDY  = {"PK","single-dose tox","repeat-dose tox","immunogenicity","biodistribution","efficacy","TCR"}
ENUM_JUSTIF = {"pharmacological relevance","regulatory expectation","historical precedent","not stated"}
ENUM_CR     = {"in-vitro binding shown","sequence homology only","not addressed"}
SKIP = {"eval.json","eval_script.json","eval_llm.json","score.json"}

rows = []
for jp in sorted(glob.glob("stage_05/data/*.json")):
    if os.path.basename(jp) in SKIP: continue
    rec = json.load(open(jp))
    arms = rec.get("animal_arms") or []
    enums_ok = (
        rec.get("format") in ENUM_FORMAT | {None}
        and rec.get("development_stage") in ENUM_STAGE | {None}
        and rec.get("regulatory_context") in ENUM_REG | {None}
        and all(a.get("species") in ENUM_SPECIES | {None}
                and a.get("study_type") in ENUM_STUDY | {None}
                and a.get("species_justification") in ENUM_JUSTIF | {None}
                and a.get("cross_reactivity_evidence") in ENUM_CR | {None}
                for a in arms)
    )
    numerics_ok = all(
        (a.get("n_animals") is None or (isinstance(a.get("n_animals"), int) and a["n_animals"] >= 0))
        and (a.get("duration_days") is None or (isinstance(a.get("duration_days"), int) and a["duration_days"] >= 0))
        for a in arms
    )
    rows.append({
        "stem": os.path.splitext(os.path.basename(jp))[0],
        "pmid": rec.get("pmid"),
        "has_required_keys": bool(rec.get("pmid")) and isinstance(arms, list),
        "source_type_valid": rec.get("source_type") in ENUM_SOURCE,
        "enums_within_vocab": enums_ok,
        "numerics_non_negative": numerics_ok,
    })

print("Script checks (per paper):")
for r in rows:
    flags = " ".join("OK  " if r[k] else "FAIL"
        for k in ("has_required_keys","source_type_valid","enums_within_vocab","numerics_non_negative"))
    print(f"  {r['stem']:18s} keys/src/enums/numerics  {flags}")

with open("stage_05/eval/eval_script.json", "w") as f:
    json.dump({"script_per_paper": rows}, f, indent=2)
KEYS = ("has_required_keys","source_type_valid","enums_within_vocab","numerics_non_negative")
n_pass = sum(1 for r in rows for k in KEYS if r[k])
n_total = len(rows) * len(KEYS)
score_path = "stage_05/eval/score.json"
score = json.load(open(score_path)) if os.path.exists(score_path) else {}
score["script"] = {"passed": n_pass, "total": n_total,
                   "percent": round(100*n_pass/n_total, 1) if n_total else 0.0}
with open(score_path, "w") as f: json.dump(score, f, indent=2)
print(f"\nScore: {n_pass}/{n_total} ({score['script']['percent']}%)")
print("\nOptional AI-grader (costs ~$0.10):")
print("  !python eval/eval_05_llm.py")


## 9. Stuck? Skip this stage


Copy the reference run's Stage 5 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil, glob
os.makedirs("stage_05/data", exist_ok=True)
for src in glob.glob("reference_outputs/stage_05/data/*.json"):
    shutil.copy(src, "stage_05/data/")
print(f"copied {len(os.listdir('stage_05/data'))} reference extractions")
